# Stage 4: ERAP2 Binder Design with Proteina-Complexa

Complete self-contained notebook. Run all cells top to bottom.

**Target:** ERAP2 active site (H370, E371, H374, E393)
**Tool:** Proteina-Complexa (NVIDIA) — 68% hit rate, picomolar binders

In [ ]:
# Cell 1: Install build tools + clone repo
!pip install -q hatchling
import os
os.chdir("/content")
if not os.path.exists("proteina-complexa"):
    !git clone https://github.com/NVIDIA-Digital-Bio/proteina-complexa.git
print("Repo ready.")

In [ ]:
# Cell 2: Install dependencies manually (bypass broken pip install -e)
!pip install -q python-dotenv==1.0.1 einops==0.6 dm-tree==0.1.8 loguru==0.7.2 \
    tqdm==4.66.4 joblib==1.4.2 rich==14.0.0 omegaconf pydantic rich-click \
    multipledispatch wget==3.2 hydra-core==1.3.1 ml-collections==0.1.1 \
    jaxtyping loralib scipy==1.12.0 h5py xarray deepdiff \
    biopandas==0.5.1 biopython biotite cpdb-protein==0.2.0 \
    mdtraj==1.10.2 ProDy==2.6.1 wandb==0.23.1 plotly seaborn scikit-learn

# Lightning — needs specific version range
!pip install -q "lightning>=2.5.0,<2.6"

# Transformers — they want >=5.3.0 but let's check what's available
!pip install -q --upgrade transformers

print("Dependencies installed.")

In [ ]:
# Cell 3: Add src to path so we can import proteinfoundation
import sys
sys.path.insert(0, "/content/proteina-complexa/src")
sys.path.insert(0, "/content/proteina-complexa")

# Verify import
try:
    import proteinfoundation
    print(f"proteinfoundation imported successfully")
except ImportError as e:
    print(f"Import error: {e}")
    print("Will use CLI script directly instead.")

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 4: Download model weights from NGC
import os, subprocess
os.chdir("/content/proteina-complexa")
os.makedirs("ckpts", exist_ok=True)

NGC_BASE = "https://api.ngc.nvidia.com/v2/models/org/nvidia/team/clara/proteina_complexa/1.0/files?redirect=true&path="

weights = {
    "ckpts/complexa.ckpt": "complexa.ckpt",
    "ckpts/complexa_ae.ckpt": "complexa_ae.ckpt",
}

for local_path, remote_name in weights.items():
    if os.path.exists(local_path) and os.path.getsize(local_path) > 1000:
        print(f"Already have {local_path} ({os.path.getsize(local_path)/1e6:.0f} MB)")
    else:
        print(f"Downloading {remote_name}...")
        url = NGC_BASE + remote_name
        !wget -q --show-progress -O {local_path} "{url}"
        size = os.path.getsize(local_path) if os.path.exists(local_path) else 0
        print(f"  Saved: {size/1e6:.0f} MB")

print("\nWeight files:")
!ls -lh ckpts/*.ckpt

In [ ]:
# Cell 5: Download ERAP2 structure + create target config
import requests, yaml, shutil
os.chdir("/content/proteina-complexa")

# Download ERAP2 PDB from AlphaFold
target_dir = "assets/target_data/erap2"
os.makedirs(target_dir, exist_ok=True)
pdb_path = f"{target_dir}/ERAP2.pdb"

if not os.path.exists(pdb_path):
    resp = requests.get("https://alphafold.ebi.ac.uk/files/AF-Q6P179-F1-model_v6.pdb")
    resp.raise_for_status()
    with open(pdb_path, "w") as f:
        f.write(resp.text)
    print(f"Downloaded ERAP2 structure: {len(resp.text)} chars")
else:
    print(f"Already have {pdb_path}")

# Add ERAP2 to targets_dict.yaml
targets_file = "configs/targets/targets_dict.yaml"
with open(targets_file) as f:
    targets = yaml.safe_load(f)

# ERAP2 is 960 residues, chain A
# Active site + variant hotspots
targets["target_dict_cfg"]["ERAP2"] = {
    "source": "erap2",
    "target_filename": "ERAP2",
    "target_path": f"./assets/target_data/erap2/ERAP2.pdb",
    "target_input": "A1-960",
    "hotspot_residues": ["A370", "A371", "A374", "A392", "A393"],
    "binder_length": [60, 100],
    "pdb_id": None,
}

with open(targets_file, "w") as f:
    yaml.dump(targets, f, default_flow_style=False)

print("ERAP2 target config added:")
print(f"  PDB: {pdb_path}")
print(f"  Chain: A, Residues: 1-960")
print(f"  Hotspots: A370(H-Zn), A371(E-cat), A374(H-Zn), A392(K-variant), A393(E-Zn)")
print(f"  Binder lengths: 60-100 residues")

In [ ]:
# Cell 6: Run binder design
import sys
os.chdir("/content/proteina-complexa")
sys.path.insert(0, "/content/proteina-complexa/src")

# Run via the CLI runner script directly
!python -c "
import sys
sys.path.insert(0, 'src')
sys.argv = [
    'complexa', 'design',
    'configs/search_binder_local_pipeline.yaml',
    '++run_name=erap2_binders',
    '++generation.task_name=ERAP2',
    '++generation.num_samples=20',
    '++seed=42',
]
from proteinfoundation.cli.cli_runner import main
main()
"

In [ ]:
# Cell 7: Analyze results
import glob, os
import pandas as pd
os.chdir("/content/proteina-complexa")

# Find output files
for search_dir in ["inference", "outputs", "evaluation_results", "logs"]:
    if os.path.exists(search_dir):
        print(f"=== {search_dir}/ ===")
        for root, dirs, files in os.walk(search_dir):
            for f in files:
                fp = os.path.join(root, f)
                size = os.path.getsize(fp)
                if size > 100:  # skip empty files
                    print(f"  {fp} ({size/1024:.1f} KB)")

# Load any CSV results
csvs = glob.glob("inference/**/*.csv", recursive=True) + \
       glob.glob("outputs/**/*.csv", recursive=True) + \
       glob.glob("evaluation_results/**/*.csv", recursive=True)

pdbs = glob.glob("inference/**/*.pdb", recursive=True) + \
       glob.glob("outputs/**/*.pdb", recursive=True)

print(f"\nTotal CSV files: {len(csvs)}")
print(f"Total PDB files: {len(pdbs)}")

if csvs:
    for csv_path in csvs[:3]:
        print(f"\n--- {csv_path} ---")
        df = pd.read_csv(csv_path)
        print(f"  {len(df)} rows x {len(df.columns)} columns")
        print(f"  Columns: {list(df.columns)}")
        if len(df) > 0:
            print(df.head(10).to_string())

if pdbs:
    print(f"\nGenerated {len(pdbs)} binder structures:")
    for p in sorted(pdbs)[:10]:
        print(f"  {p}")

if not csvs and not pdbs:
    print("\nNo results yet. Check cell 6 output for errors.")
    print("\nLooking for any output files...")
    !find /content/proteina-complexa -name '*.pdb' -newer /content/proteina-complexa/README.md 2>/dev/null | head -20
    !find /content/proteina-complexa -name '*.csv' -newer /content/proteina-complexa/README.md 2>/dev/null | head -20